In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
from google.colab import userdata
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mtick

In [ ]:
DB_USER = userdata.get('DB_USER')
DB_PASSWORD = userdata.get('DB_PASSWORD')
DB_HOST = userdata.get('DB_HOST')
DB_NAME = userdata.get('DB_NAME')
DB_PORT = "5432"

In [ ]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sqlalchemy.create_engine(connection_string)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
all_data_so_far = """
SELECT
    timestamp,
    gateway_serial,
    "total_system_kWh",
    active_assets,
    active_asset_count,
    dominant_asset,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    input_channel_1_current,
    input_channel_2_current,
    input_channel_3_current,
    input_channel_4_current,
    input_channel_5_current,
    input_channel_6_current
FROM public.smart_device_readings
ORDER BY timestamp DESC;
"""

all_data_df = pd.read_sql(all_data_so_far, engine)

In [ ]:
input_channel_columns = [
    'input_channel_1_current',
    'input_channel_2_current',
    'input_channel_3_current',
    'input_channel_4_current',
    'input_channel_5_current',
    'input_channel_6_current'
]

# Ensure columns exist before attempting to find the max
existing_input_channels = [col for col in input_channel_columns if col in all_data_df.columns]

if existing_input_channels:
    max_input_current = all_data_df[existing_input_channels].max().max()
    print(f"The maximum input channel current is: {max_input_current}")
else:
    print("No input channel current columns found in the DataFrame.")

In [ ]:
max_input_channel_query = """
SELECT
    MAX(GREATEST(
        input_channel_1_current,
        input_channel_2_current,
        input_channel_3_current,
        input_channel_4_current,
        input_channel_5_current,
        input_channel_6_current
    )) AS max_overall_input_current
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515';
"""

max_input_channel_df = pd.read_sql(max_input_channel_query, engine)
display(max_input_channel_df)

In [ ]:
max_input_channel_query = """
SELECT
    MAX(GREATEST(
        input_channel_1_current,
        input_channel_2_current,
        input_channel_3_current,
        input_channel_4_current,
        input_channel_5_current,
        input_channel_6_current
    )) AS max_overall_input_current
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM21120502';
"""

max_input_channel_df = pd.read_sql(max_input_channel_query, engine)
display(max_input_channel_df)

In [ ]:
max_input_channel_query = """
SELECT
    timestamp,
    GREATEST(
        input_channel_1_current,
        input_channel_2_current,
        input_channel_3_current,
        input_channel_4_current,
        input_channel_5_current,
        input_channel_6_current
    ) AS max_overall_input_current
FROM public.smart_device_readings
ORDER BY max_overall_input_current DESC
LIMIT 1;
"""

max_input_channel_df = pd.read_sql(max_input_channel_query, engine)
display(max_input_channel_df)

In [ ]:
all_data = """
SELECT
    timestamp,
        input_channel_1_current,
        input_channel_2_current,
        input_channel_3_current,
        input_channel_4_current,
        input_channel_5_current,
        input_channel_6_current
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515'
ORDER BY timestamp DESC;
"""

all_data_df = pd.read_sql(all_data, engine)
display(all_data_df)

In [ ]:
channel_cols = [
    'input_channel_1_current',
    'input_channel_2_current',
    'input_channel_3_current',
    'input_channel_4_current',
    'input_channel_5_current',
    'input_channel_6_current'
]

# 3. Filter where ANY of those columns are > 90
filtered_df = all_data_df[all_data_df[channel_cols].gt(90).any(axis=1)]

# View the result
display(filtered_df)

In [ ]:
filtered_df.to_csv('high_current.csv', index=False)

In [ ]:
date_range_query = """
SELECT
    timestamp,
    input_channel_1_current,
    input_channel_2_current,
    input_channel_3_current,
    input_channel_4_current,
    input_channel_5_current,
    input_channel_6_current
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp >= '2026-05-17' AND timestamp < '2026-05-24'
ORDER BY timestamp ASC;
"""

date_range_df = pd.read_sql(date_range_query, engine)
display(date_range_df.head())

In [ ]:
date_range_df.sample(10)

Now, let's calculate the minimum, maximum, and average for each input channel.

In [ ]:
channel_cols = [
    'input_channel_1_current',
    'input_channel_2_current',
    'input_channel_3_current',
    'input_channel_4_current',
    'input_channel_5_current',
    'input_channel_6_current'
]

# Calculate minimums, replacing 0 with NaN for filtering
min_currents = date_range_df[channel_cols].replace(0, np.nan).min()
max_currents = date_range_df[channel_cols].max()
avg_currents = date_range_df[channel_cols].mean()
avg_non_zero_currents = date_range_df[channel_cols].replace(0, np.nan).mean()

summary_df = pd.DataFrame({
    'Minimum Current (excluding zeros)': min_currents,
    'Maximum Current': max_currents,
    'Average Current': avg_currents,
    'Average Non-Zero Current': avg_non_zero_currents
})

display(summary_df)

In [ ]:
summary_df.to_csv('current_17_23_may.csv', index=False)

### Active Power Overall Total Analysis

Now, let's analyze the `active_power_overall_total` for the same date range (17th - 23rd of May).

In [ ]:
active_power_query = """
SELECT
    timestamp,
    active_power_overall_total
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp >= '2026-05-17' AND timestamp < '2026-05-24'
ORDER BY timestamp ASC;
"""

active_power_df = pd.read_sql(active_power_query, engine)
display(active_power_df.head())

In [ ]:
power_col = 'active_power_overall_total'

# Calculate minimum value strictly greater than 0
min_power_gt_zero = active_power_df[active_power_df[power_col] > 0][power_col].min()
max_power = active_power_df[power_col].max()
avg_power = active_power_df[power_col].mean()
avg_non_zero_power = active_power_df[active_power_df[power_col] != 0][power_col].mean()

power_summary_df = pd.DataFrame({
    'Metric': [
        'Minimum Active Power (> 0)',
        'Maximum Active Power',
        'Average Active Power',
        'Average Non-Zero Active Power'
    ],
    'Value': [min_power_gt_zero, max_power, avg_power, avg_non_zero_power]
})

display(power_summary_df)

In [ ]:
power_col = 'active_power_overall_total'

# Original absolute minimum
absolute_min = active_power_df[power_col].min()

# Filtered values
min_power_gt_zero = active_power_df[active_power_df[power_col] > 0][power_col].min()
max_power = active_power_df[power_col].max()
avg_power = active_power_df[power_col].mean()
avg_non_zero_power = active_power_df[active_power_df[power_col] != 0][power_col].mean()

# Combined Summary
combined_power_summary = pd.DataFrame({
    'Metric': [
        'Absolute Minimum Active Power',
        'Minimum Active Power (> 0)',
        'Maximum Active Power',
        'Average Active Power',
        'Average Non-Zero Active Power'
    ],
    'Value': [absolute_min, min_power_gt_zero, max_power, avg_power, avg_non_zero_power]
})

display(combined_power_summary)

In [ ]:
combined_power_summary.to_csv('load_17_23_may.csv', index=False)

In [ ]:
import os

# Define the folder path where your files are located
folder_path = '/content/drive/Shareddrives/Data Team/ABMF Data/Historical Data & Backup/Branches Data/Head Office/2026'

# List all files in the folder
file_names = [f for f in os.listdir(folder_path)]

# Initialize an empty list to store individual DataFrames
all_dfs = []

# Loop through each file, read it, and append to the list
for file in file_names:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(file_path)
        all_dfs.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Concatenate all DataFrames into a single one
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Sort the combined DataFrame by the 'Time' column
    # Assuming 'Time' column exists and is parsable. Adjust if column name is different.
    if 'Time' in combined_df.columns:
        combined_df['Time'] = pd.to_datetime(combined_df['Time'], errors='coerce')
        combined_df = combined_df.sort_values(by='Time').reset_index(drop=True)
        print("Successfully combined and sorted the files.")
        display(combined_df.head())
    else:
        print("The 'Time' column was not found for sorting.")
        display(combined_df.head())
else:
    print(f"No CSV files found in the folder: {folder_path}")

### Analysis for Apr 26th - May 2nd, 2026
Filtering `combined_df` for the specific date range and calculating metrics for currents and power.

In [ ]:
# 1. Filter the DataFrame for the date range
# Note: End date is exclusive to capture until the end of May 2nd
start_date = '2026-04-26'
end_date = '2026-05-03'

mask = (combined_df['Time'] >= start_date) & (combined_df['Time'] < end_date)
filtered_combined_df = combined_df.loc[mask].copy()

print(f"Rows found for the range: {len(filtered_combined_df)}")
display(filtered_combined_df.head() if not filtered_combined_df.empty else "No data found")

In [ ]:
# 2. Current metrics for specific columns
current_cols = ['I001_A', 'I002_A', 'I004_A', 'I005_A', 'I007_A', 'I008_A']

# Filter existing columns only to prevent errors
existing_cols = [col for col in current_cols if col in filtered_combined_df.columns]

if existing_cols:
    min_currents = filtered_combined_df[existing_cols].replace(0, np.nan).min()
    max_currents = filtered_combined_df[existing_cols].max()
    avg_currents = filtered_combined_df[existing_cols].mean()
    avg_non_zero_currents = filtered_combined_df[existing_cols].replace(0, np.nan).mean()

    current_summary_df = pd.DataFrame({
        'Minimum Current (excluding zeros)': min_currents,
        'Maximum Current': max_currents,
        'Average Current': avg_currents,
        'Average Non-Zero Current': avg_non_zero_currents
    })

    print("Current Metrics (Apr 26 - May 2):")
    display(current_summary_df)
else:
    print("Requested current columns not found in the dataset.")

In [ ]:
current_summary_df.to_csv('current_26_2.csv')

In [ ]:
# 3. Power metrics for P_kW
power_col_new = 'P_kW'

if power_col_new in filtered_combined_df.columns:
    abs_min_p = filtered_combined_df[power_col_new].min()
    min_p_gt_zero = filtered_combined_df[filtered_combined_df[power_col_new] > 0][power_col_new].min()
    max_p = filtered_combined_df[power_col_new].max()
    avg_p = filtered_combined_df[power_col_new].mean()
    avg_non_zero_p = filtered_combined_df[filtered_combined_df[power_col_new] != 0][power_col_new].mean()

    power_summary_new = pd.DataFrame({
        'Metric': [
            'Absolute Minimum P_kW',
            'Minimum P_kW (> 0)',
            'Maximum P_kW',
            'Average P_kW',
            'Average Non-Zero P_kW'
        ],
        'Value': [abs_min_p, min_p_gt_zero, max_p, avg_p, avg_non_zero_p]
    })

    print("Power Metrics (Apr 26 - May 2):")
    display(power_summary_new)
else:
    print(f"Column '{power_col_new}' not found in the dataset.")

In [ ]:
power_summary_new.to_csv('load_26_2.csv', index=False)